# 实验一：队员选拔问题 - 规划模型

## 实验题目

某游泳队准备从 5 名队员中选择 4 人组成接力队，参加 4×100 米混合泳接力比赛。每名队员只能承担一种泳姿，每种泳姿必须由一名队员完成。目标是在给定成绩数据下，使接力总用时最短。


## 问题背景

混合泳接力包含蝶泳、仰泳、蛙泳和自由泳四个项目。由于同一名队员在不同泳姿上的成绩差异较大，直接选择每个泳姿最快的人可能会出现同一人被重复安排的问题。因此，需要把该问题看作一个分配规划问题，在“每种泳姿恰好一人”和“每名队员最多参加一种泳姿”的约束下，寻找总成绩最优的接力队方案。

本实验还考虑近期训练成绩变化：丁的蛙泳退步至 1’15’’2，戊的自由泳达到 57’’5。通过重新求解模型，可以判断选拔方案是否需要调整。


## 数学模型建立

设队员集合为 $I=\{甲,乙,丙,丁,戊\}$，泳姿集合为 $J=\{蝶泳,仰泳,蛙泳,自由泳\}$。设 $t_{ij}$ 表示队员 $i$ 完成泳姿 $j$ 的用时，决策变量为：

$$x_{ij}=\begin{cases}1, & \text{队员 }i\text{ 被分配到泳姿 }j\\0, & \text{否则}\end{cases}$$

目标函数为总用时最短：

$$\min \sum_{i\in I}\sum_{j\in J} t_{ij}x_{ij}$$

约束条件为：

每种泳姿恰好由 1 名队员完成：

$$\sum_{i\in I}x_{ij}=1,\quad j\in J$$

每名队员最多参加 1 种泳姿：

$$\sum_{j\in J}x_{ij}\le 1,\quad i\in I$$

变量取值为 0 或 1：

$$x_{ij}\in\{0,1\}$$

这是一个典型的 0-1 分配规划模型。


## 算法设计思路

本题只有 5 名队员和 4 种泳姿。若按泳姿顺序依次安排不同队员，总可行方案数为：

$$P_5^4=5\times4\times3\times2=120$$

因此可以使用穷举法遍历全部可行分配方案，并选出总用时最短的一组。穷举法在本题规模下能够保证得到全局最优解，且便于观察模型约束如何被程序实现。


In [ ]:
from pathlib import Path
from itertools import permutations
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = Path.cwd().parent / 'outputs' if Path.cwd().name == 'notebooks' else Path('math_modeling_experiments/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Python 程序实现

### 1. 原始成绩录入

先把题目中的成绩以字符串形式录入。字符串中的 `1’06’’8` 表示 1 分 06.8 秒，`57’’2` 表示 57.2 秒。


In [ ]:
swimmers = ['甲', '乙', '丙', '丁', '戊']
strokes = ['蝶泳', '仰泳', '蛙泳', '自由泳']

raw_times = pd.DataFrame(
    [
        ['1’06’’8', '57’’2', '1’18’’', '1’10’’', '1’07’’4'],
        ['1’15’’6', '1’06’’', '1’07’’8', '1’14’’2', '1’11’’'],
        ['1’27’’', '1’06’’4', '1’24’’6', '1’09’’6', '1’23’’8'],
        ['58’’6', '53’’', '59’’4', '57’’2', '1’02’’4'],
    ],
    index=strokes,
    columns=swimmers,
)

raw_times


### 2. 成绩字符串转换为秒数

建立一个转换函数，把分钟和秒的字符串统一转为秒数，便于后续计算目标函数。


In [ ]:
def parse_swim_time(value):
    """将类似 1’06’’8、57’’2、53’’ 的成绩字符串转换为秒。"""
    text = str(value).strip()
    text = text.replace('′', '’').replace("'", '’').replace('＇', '’')
    text = text.replace('″', '’’').replace('”', '’’').replace('"', '’’')

    if '’’' in text:
        main_part, decimal_part = text.split('’’', 1)
    else:
        main_part, decimal_part = text, ''

    if '’' in main_part:
        minute_part, second_part = main_part.split('’', 1)
        minutes = int(minute_part)
    else:
        minutes = 0
        second_part = main_part

    seconds = int(second_part) if second_part else 0
    decimal = int(decimal_part) / (10 ** len(decimal_part)) if decimal_part else 0.0
    return minutes * 60 + seconds + decimal


time_seconds = raw_times.map(parse_swim_time)
time_seconds


### 3. 建立穷举求解函数

程序中每个排列表示按照 `蝶泳、仰泳、蛙泳、自由泳` 的顺序分别安排哪位队员。由于 `permutations(swimmers, 4)` 会自动保证 4 个位置上的队员互不相同，所以已经满足“每名队员最多参加一种泳姿”的约束。


In [ ]:
def solve_relay_assignment(time_df):
    """求解 5 人选 4 人的混合泳接力最优分配方案。"""
    best_assignment = None
    best_total_time = np.inf

    for assigned_swimmers in permutations(time_df.columns, len(time_df.index)):
        total_time = sum(
            time_df.loc[stroke, swimmer]
            for stroke, swimmer in zip(time_df.index, assigned_swimmers)
        )
        if total_time < best_total_time:
            best_assignment = assigned_swimmers
            best_total_time = total_time

    result = pd.DataFrame({
        '泳姿': list(time_df.index),
        '队员': list(best_assignment),
        '成绩秒': [time_df.loc[stroke, swimmer] for stroke, swimmer in zip(time_df.index, best_assignment)],
    })
    result['成绩秒'] = result['成绩秒'].round(1)

    selected = set(best_assignment)
    not_selected = [name for name in time_df.columns if name not in selected]
    return result, round(float(best_total_time), 1), not_selected


original_solution, original_total, original_unselected = solve_relay_assignment(time_seconds)
original_solution


## 结果输出：原始成绩下的最优接力队方案

下面输出原始成绩下的最优分配结果、总成绩和未入选队员。


In [ ]:
print('原始成绩下的总时间：', original_total, '秒')
print('原始成绩下未入选队员：', '、'.join(original_unselected))

expected_original = {
    '蝶泳': ('乙', 57.2),
    '仰泳': ('丙', 67.8),
    '蛙泳': ('丁', 69.6),
    '自由泳': ('甲', 58.6),
}
assert original_total == 253.2
for _, row in original_solution.iterrows():
    expected_swimmer, expected_time = expected_original[row['泳姿']]
    assert row['队员'] == expected_swimmer
    assert row['成绩秒'] == expected_time

original_solution


## 更新成绩后重新求解

近期训练显示：丁的蛙泳退步至 1’15’’2，戊的自由泳达到 57’’5。下面更新成绩表并重新求解最优方案。


In [ ]:
updated_raw_times = raw_times.copy()
updated_raw_times.loc['蛙泳', '丁'] = '1’15’’2'
updated_raw_times.loc['自由泳', '戊'] = '57’’5'

updated_time_seconds = updated_raw_times.map(parse_swim_time)
updated_solution, updated_total, updated_unselected = solve_relay_assignment(updated_time_seconds)

print('更新成绩后的总时间：', updated_total, '秒')
print('更新成绩后未入选队员：', '、'.join(updated_unselected))

expected_updated = {
    '蝶泳': ('乙', 57.2),
    '仰泳': ('丙', 67.8),
    '蛙泳': ('丁', 75.2),
    '自由泳': ('戊', 57.5),
}
assert updated_total == 257.7
for _, row in updated_solution.iterrows():
    expected_swimmer, expected_time = expected_updated[row['泳姿']]
    assert row['队员'] == expected_swimmer
    assert row['成绩秒'] == expected_time

updated_solution


## 两次结果比较

将原始方案和更新后方案放在同一张表中，观察每个泳姿的队员安排和成绩变化。


In [ ]:
comparison = original_solution.merge(
    updated_solution,
    on='泳姿',
    suffixes=('（原始）', '（更新后）')
)
comparison['是否调整'] = np.where(
    comparison['队员（原始）'] == comparison['队员（更新后）'],
    '不变',
    '调整'
)
comparison.loc[len(comparison)] = [
    '总时间', '', original_total, '', updated_total,
    '增加 {:.1f} 秒'.format(updated_total - original_total)
]
comparison.loc[len(comparison)] = [
    '未入选队员', '、'.join(original_unselected), '', '、'.join(updated_unselected), '', '调整'
]
comparison


## 图表或表格展示

除上方对比表外，下面用柱状图展示原始方案与更新后方案在各泳姿上的成绩差异，并保存到 `outputs/` 文件夹。


In [ ]:
plot_df = pd.DataFrame({
    '泳姿': strokes,
    '原始方案成绩秒': original_solution['成绩秒'].to_numpy(),
    '更新后方案成绩秒': updated_solution['成绩秒'].to_numpy(),
}).set_index('泳姿')

ax = plot_df.plot(kind='bar', figsize=(9, 5), width=0.72)
ax.set_title('原始方案与更新后方案成绩比较')
ax.set_ylabel('用时（秒）')
ax.set_xlabel('泳姿')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()

fig_path = OUTPUT_DIR / '01_relay_selection_comparison.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('图表已保存：', fig_path)


## 结论分析

原始成绩下，最优接力队方案为：蝶泳乙、仰泳丙、蛙泳丁、自由泳甲，总时间为 253.2 秒，未入选队员为戊。

更新成绩后，丁的蛙泳成绩从 69.6 秒变为 75.2 秒，戊的自由泳成绩提高到 57.5 秒。重新求解后，最优方案变为：蝶泳乙、仰泳丙、蛙泳丁、自由泳戊，总时间为 257.7 秒，未入选队员变为甲。

因此，选拔结果应该调整：自由泳由甲调整为戊，未入选队员由戊变为甲。虽然总时间比原始成绩下增加 4.5 秒，但这是在近期训练成绩变化后的最优可行安排。
